In [ ]:
# Cell 1: Clone repo, install dependencies, and patch pytorchvideo compatibility
!git clone https://github.com/dharineesh730-lab/surviellanceSystem.git /content/surviellanceSystem
%cd /content/surviellanceSystem
!pip install -r requirements.txt
!pip install pyngrok

# Fix: pytorchvideo imports torchvision.transforms.functional_tensor which was removed
# in newer torchvision. Patch the source file directly.
!find /usr -name 'augmentations.py' -path '*/pytorchvideo/*' \
    -exec sed -i 's|import torchvision.transforms.functional_tensor as F_t|import torchvision.transforms.functional as F_t|g' {} \;
!find /usr -name 'functional.py' -path '*/pytorchvideo/*' \
    -exec sed -i 's|import torchvision.transforms.functional_tensor as F_t|import torchvision.transforms.functional as F_t|g' {} \;
print('pytorchvideo patch applied.')

In [ ]:
# Cell 2: Upload ckpt.t7 model weight (do this ONCE per session)
from google.colab import files
import os

os.makedirs('models', exist_ok=True)
uploaded = files.upload()  # select ckpt.t7 from your computer
for filename in uploaded:
    dest = os.path.join('models', os.path.basename(filename))
    os.rename(filename, dest)
print('Model ready:', os.listdir('models'))

In [ ]:
# Cell 3: Launch Flask web UI via ngrok
# Get your free auth token from: https://dashboard.ngrok.com/get-started/your-authtoken
# Paste it below between the quotes
NGROK_TOKEN = '3Ah3ScDXd7qw9TuTy74YKwddTJd_ttmDnP8PY6KFVh6ebpqY'

import subprocess
import socket
import time
import threading
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_TOKEN)

# Start Flask and stream its output so errors are visible
flask_proc = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def stream_output():
    for line in flask_proc.stdout:
        print('[Flask]', line, end='')

threading.Thread(target=stream_output, daemon=True).start()

# Wait until Flask is actually listening on port 5001 (up to 120 seconds)
started = False
print('Waiting for Flask to start', end='')
for i in range(120):
    if flask_proc.poll() is not None:
        print('\nFlask exited unexpectedly — see [Flask] lines above for the error.')
        break
    try:
        with socket.create_connection(('localhost', 5001), timeout=1):
            started = True
            break
    except OSError:
        time.sleep(1)
        print('.', end='', flush=True)

if started:
    public_url = ngrok.connect(5001)
    print('\n' + '=' * 50)
    print('  Open this URL in your browser:')
    print(' ', public_url)
    print('=' * 50)
    print('Upload videos through the web UI — no Colab upload needed.')
else:
    print('Flask did not start within 120 seconds.')